# A_S3.1 — Permutation Test Results Visualization

Analyzes the output of A_S3 (permutation test) to evaluate the testing framework.

| Section | Question |
|---------|----------|
| 1. Classification overview | Overall detection rates by category |
| 2. p-value uniformity | True Null p-values should be Uniform(0,1) |
| 3. FPR calibration | Is false positive rate ≤ α under True Null? |
| 4. Null distribution examples | Shape of null vs observed for selected cases |
| 5. Detection by SNR | How does signal strength affect detection? |
| 6. Detection by function family | Which functions are hard to detect? |
| 7. Metric contribution | Which metrics drive the joint test? |
| 8. Variance-only detection | Can the framework detect spread-only dependence? |
| 9. Metric ablation | How much does each metric group contribute? |

In [ ]:
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
import seaborn as sns
%matplotlib inline

S1_DIR = Path('output/S1')
S3_DIR = Path('output/S3')
VIZ_DIR = Path('output/S3/viz')
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Load case metadata
cases_main = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
cases_null = pd.read_csv(S1_DIR / 'null_expanded_cases.csv', low_memory=False)
cases_main['source'] = 'main'
cases_null['source'] = 'null_expanded'
offset = cases_main['case_id'].max()
cases_null['case_id'] = cases_null['case_id'] + offset
cases_df = pd.concat([cases_main, cases_null], ignore_index=True)

is_null = cases_df['family_id'] == 'Null'
is_const = cases_df['spread_pattern'] == 'constant'
cases_df['category'] = 'mean+variance'
cases_df.loc[is_null & is_const, 'category'] = 'true_null'
cases_df.loc[~is_null & is_const, 'category'] = 'mean_only'
cases_df.loc[is_null & ~is_const, 'category'] = 'variance_only'

# Load permutation results
perm = pd.read_parquet(S3_DIR / 'permutation_all.parquet')
df = cases_df.merge(perm, on='case_id', suffixes=('', '_p'))

print(f'Loaded {len(df):,} cases with permutation results')
print(df['category'].value_counts())

In [ ]:
CAT_ORDER = ['true_null', 'variance_only', 'mean_only', 'mean+variance']
CAT_COLORS = {'true_null': '#999999', 'variance_only': '#f59e0b',
              'mean_only': '#3b82f6', 'mean+variance': '#10b981'}
CAT_LABELS = {'true_null': 'True Null', 'variance_only': 'Variance-only',
              'mean_only': 'Mean-only', 'mean+variance': 'Mean+Variance'}

ALPHA = 0.05

## 1. Classification Overview

In [ ]:
# Detection rate by category
det_rate = df.groupby('category').apply(
    lambda g: pd.Series({
        'n': len(g),
        'detectable': (g['classification'] == 'detectable').sum(),
        'not_detectable': (g['classification'] == 'not_detectable').sum(),
        'uncertain': (g['classification'] == 'uncertain').sum(),
        'detection_rate': (g['classification'] == 'detectable').mean(),
    })
).loc[CAT_ORDER]

display(det_rate)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of detection rates
ax = axes[0]
bars = ax.bar(range(len(CAT_ORDER)), det_rate['detection_rate'],
              color=[CAT_COLORS[c] for c in CAT_ORDER])
ax.set_xticks(range(len(CAT_ORDER)))
ax.set_xticklabels([CAT_LABELS[c] for c in CAT_ORDER], rotation=20, ha='right')
ax.set_ylabel('Detection Rate')
ax.set_title('Detection Rate by Category')
ax.set_ylim(0, 1.05)
for bar, rate in zip(bars, det_rate['detection_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{rate:.1%}', ha='center', fontsize=10)

# Stacked bar chart of classification
ax = axes[1]
bottom = np.zeros(len(CAT_ORDER))
for cls, color in [('detectable', '#22c55e'), ('uncertain', '#facc15'),
                    ('not_detectable', '#ef4444')]:
    vals = det_rate[cls].values / det_rate['n'].values
    ax.bar(range(len(CAT_ORDER)), vals, bottom=bottom, color=color, label=cls)
    bottom += vals
ax.set_xticks(range(len(CAT_ORDER)))
ax.set_xticklabels([CAT_LABELS[c] for c in CAT_ORDER], rotation=20, ha='right')
ax.set_ylabel('Fraction')
ax.set_title('Classification Breakdown')
ax.legend(fontsize=8)

plt.suptitle('Permutation Test — Classification Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig1_classification_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. p-Value Uniformity Under True Null

If the test is well-calibrated, True Null p-values should follow Uniform(0,1).
Deviation indicates either conservatism (p > expected) or anti-conservatism (p < expected).

In [ ]:
tn = df[df['category'] == 'true_null'].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogram
ax = axes[0]
ax.hist(tn['p_value'], bins=50, density=True, color='#999999', alpha=0.7, edgecolor='white')
ax.axhline(1.0, color='red', ls='--', lw=1.5, label='Uniform(0,1)')
ax.set_xlabel('p-value')
ax.set_ylabel('Density')
ax.set_title('True Null p-Value Distribution')
ax.legend()

# QQ plot
ax = axes[1]
p_sorted = np.sort(tn['p_value'].values)
n = len(p_sorted)
expected = np.arange(1, n + 1) / (n + 1)
ax.scatter(expected, p_sorted, s=2, alpha=0.3, color='#999999')
ax.plot([0, 1], [0, 1], 'r--', lw=1.5)
ax.set_xlabel('Expected (Uniform)')
ax.set_ylabel('Observed p-value')
ax.set_title('QQ Plot')
ax.set_aspect('equal')

# Empirical CDF
ax = axes[2]
ax.plot(p_sorted, np.arange(1, n + 1) / n, color='#999999', label='Observed')
ax.plot([0, 1], [0, 1], 'r--', lw=1.5, label='Uniform')
ax.set_xlabel('p-value')
ax.set_ylabel('Cumulative Fraction')
ax.set_title('Empirical CDF')
ax.legend()

plt.suptitle(f'p-Value Calibration Under True Null (n={len(tn):,})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig2_pvalue_uniformity.png', dpi=150, bbox_inches='tight')
plt.show()

# KS test against uniform
ks_stat, ks_p = stats.kstest(tn['p_value'], 'uniform')
print(f'KS test vs Uniform(0,1): statistic={ks_stat:.4f}, p={ks_p:.4g}')
print(f'Empirical FPR at α=0.05: {(tn["p_value"] <= 0.05).mean():.4f}')
print(f'Empirical FPR at α=0.01: {(tn["p_value"] <= 0.01).mean():.4f}')

## 3. FPR Calibration

False positive rate at various α levels. Also broken down by spread pattern and x-distribution.

In [ ]:
alphas = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20]
fpr_overall = [float((tn['p_value'] <= a).mean()) for a in alphas]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall FPR vs α
ax = axes[0]
ax.plot(alphas, fpr_overall, 'o-', color='#999999', markersize=6)
ax.plot([0, 0.2], [0, 0.2], 'r--', lw=1.5, label='Ideal')
ax.set_xlabel('Nominal α')
ax.set_ylabel('Empirical FPR')
ax.set_title('FPR vs Nominal α')
ax.legend()

# FPR by spread pattern
ax = axes[1]
for sp in ['constant', 'increasing', 'decreasing', 'middle_high']:
    sub = tn[tn['spread_pattern'] == sp]
    fpr_sp = [float((sub['p_value'] <= a).mean()) for a in alphas]
    ax.plot(alphas, fpr_sp, 'o-', markersize=4, label=sp)
ax.plot([0, 0.2], [0, 0.2], 'r--', lw=1)
ax.set_xlabel('Nominal α')
ax.set_ylabel('Empirical FPR')
ax.set_title('FPR by Spread Pattern')
ax.legend(fontsize=7)

# FPR by x-distribution
ax = axes[2]
x_dists = tn['x_distribution'].unique()
fpr_by_xdist = {}
for xd in sorted(x_dists):
    sub = tn[tn['x_distribution'] == xd]
    fpr_by_xdist[xd] = float((sub['p_value'] <= 0.05).mean())
colors = plt.cm.tab10(np.linspace(0, 1, len(fpr_by_xdist)))
bars = ax.bar(range(len(fpr_by_xdist)), list(fpr_by_xdist.values()), color=colors)
ax.axhline(0.05, color='red', ls='--', lw=1.5, label='α=0.05')
ax.set_xticks(range(len(fpr_by_xdist)))
short_labels = [xd[:8] for xd in fpr_by_xdist.keys()]
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('FPR at α=0.05')
ax.set_title('FPR by X-Distribution')
ax.legend(fontsize=8)

plt.suptitle('False Positive Rate Calibration (True Null)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig3_fpr_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Null Distribution Examples

For selected cases, show histograms of the null distribution with the observed value marked.

In [ ]:
# Pick representative z-score columns
z_cols = sorted([c for c in df.columns if c.startswith('z_')])

# Select a few interesting metrics to show null distributions
example_metrics = []
for prefix in ['abs_pearson_r', 'abs_spearman_rho', 'dcor', 'ew_bin_eta2']:
    col = f'z_{prefix}'
    obs_col = f'{prefix}_obs'
    null_med_col = f'{prefix}_null_med'
    if col in df.columns and obs_col in df.columns:
        example_metrics.append((prefix, col, obs_col, null_med_col))

if len(example_metrics) > 0:
    # Pick one strong-signal case and one true-null case
    signal_cases = df[(df['category'] == 'mean_only') & (df['snr'] == 2.0)].head(1)
    null_cases = df[df['category'] == 'true_null'].head(1)
    example_cases = pd.concat([signal_cases, null_cases])

    fig, axes = plt.subplots(len(example_cases), len(example_metrics),
                              figsize=(4.5 * len(example_metrics), 4 * len(example_cases)))
    if len(example_cases) == 1:
        axes = axes[np.newaxis, :]

    for row_i, (_, case) in enumerate(example_cases.iterrows()):
        cat = case['category']
        for col_i, (name, z_col, obs_col, null_med_col) in enumerate(example_metrics):
            ax = axes[row_i, col_i]
            z_val = case[z_col]
            obs_val = case[obs_col]
            null_med = case[null_med_col]

            ax.axvline(z_val, color='red', lw=2, label=f'Observed Z={z_val:.2f}')
            ax.axvline(0, color='gray', ls='--', lw=1)
            ax.set_xlabel('Z-score')
            ax.legend(fontsize=7)
            if col_i == 0:
                ax.set_ylabel(f'{CAT_LABELS[cat]}\ncase {int(case["case_id"])}')
            if row_i == 0:
                ax.set_title(name, fontsize=10)

    plt.suptitle('Observed Z-scores for Example Cases', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(VIZ_DIR / 'fig4_null_examples.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No matching z-score columns found')

## 5. Detection Power by SNR

For mean-only and mean+variance cases, detection rate as a function of SNR.

In [ ]:
signal = df[df['category'].isin(['mean_only', 'mean+variance'])].copy()
signal['snr_num'] = pd.to_numeric(signal['snr'], errors='coerce')
signal = signal[signal['snr_num'].notna() & np.isfinite(signal['snr_num'])]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cat, color in zip(axes, ['mean_only', 'mean+variance'],
                           [CAT_COLORS['mean_only'], CAT_COLORS['mean+variance']]):
    sub = signal[signal['category'] == cat]
    det_by_snr = sub.groupby('snr_num').apply(
        lambda g: (g['classification'] == 'detectable').mean()
    ).reset_index(name='detection_rate')

    ax.plot(det_by_snr['snr_num'], det_by_snr['detection_rate'],
            'o-', color=color, markersize=5)
    ax.axhline(ALPHA, color='red', ls=':', lw=1, label=f'α={ALPHA}')
    ax.set_xscale('log')
    ax.set_xlabel('SNR (log scale)')
    ax.set_ylabel('Detection Rate')
    ax.set_title(f'{CAT_LABELS[cat]}: Detection vs SNR')
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8)

plt.suptitle('Detection Power vs Signal-to-Noise Ratio', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig5_detection_by_snr.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Detection by Function Family

Per-family detection rate (mean-only, collapsed across SNR).

In [ ]:
mo = df[df['category'] == 'mean_only']
det_by_fam = mo.groupby('family_id').apply(
    lambda g: (g['classification'] == 'detectable').mean()
).sort_values(ascending=True).rename('detection_rate')

fig, ax = plt.subplots(figsize=(8, max(5, len(det_by_fam) * 0.3)))
colors = ['#3b82f6' if r > 0.5 else '#ef4444' for r in det_by_fam.values]
ax.barh(range(len(det_by_fam)), det_by_fam.values, color=colors, alpha=0.7)
ax.set_yticks(range(len(det_by_fam)))
ax.set_yticklabels(det_by_fam.index, fontsize=8)
ax.set_xlabel('Detection Rate (all SNR)')
ax.set_title('Mean-only Detection Rate by Function Family', fontsize=12)
ax.axvline(0.5, color='gray', ls='--', lw=1)
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig6_detection_by_family.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Metric Contribution to Joint Test

For detected cases: which metric had the highest Z-score (drove the joint T)?

In [ ]:
detected = df[df['classification'] == 'detectable'].copy()
z_cols = sorted([c for c in df.columns if c.startswith('z_')])

if len(z_cols) > 0 and len(detected) > 0:
    z_matrix = detected[z_cols].values
    max_idx = np.argmax(z_matrix, axis=1)
    driving_metric = [z_cols[i].replace('z_', '') for i in max_idx]
    detected = detected.copy()
    detected['driving_metric'] = driving_metric

    driver_counts = detected['driving_metric'].value_counts().head(20)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(len(driver_counts)), driver_counts.values, color='#3b82f6', alpha=0.7)
    ax.set_yticks(range(len(driver_counts)))
    ax.set_yticklabels(driver_counts.index, fontsize=8)
    ax.set_xlabel('# Cases Where This Metric Had Max Z')
    ax.set_title(f'Top 20 Driving Metrics in Joint Test\n({len(detected):,} detected cases)',
                 fontsize=12)
    plt.tight_layout()
    fig.savefig(VIZ_DIR / 'fig7_driving_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Breakdown by category
    for cat in ['mean_only', 'mean+variance', 'variance_only']:
        sub = detected[detected['category'] == cat]
        if len(sub) > 0:
            top5 = sub['driving_metric'].value_counts().head(5)
            print(f'\n{CAT_LABELS[cat]} — top drivers:')
            for m, c in top5.items():
                print(f'  {m:35s} {c:>5,} ({c/len(sub):.1%})')
else:
    print('No z-score columns or no detected cases found')

## 8. Variance-Only Detection

Detection rates for variance-only cases, broken down by spread pattern and x-distribution.

In [ ]:
vo = df[df['category'] == 'variance_only']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By spread pattern
ax = axes[0]
spread_patterns = ['increasing', 'decreasing', 'middle_high']
det_by_sp = []
for sp in spread_patterns:
    sub = vo[vo['spread_pattern'] == sp]
    det_by_sp.append(float((sub['classification'] == 'detectable').mean()))
bars = ax.bar(spread_patterns, det_by_sp,
              color=['#f59e0b', '#ef4444', '#8b5cf6'], alpha=0.7)
for bar, rate in zip(bars, det_by_sp):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{rate:.1%}', ha='center', fontsize=10)
ax.set_ylabel('Detection Rate')
ax.set_title('Variance-Only Detection by Spread Pattern')
ax.set_ylim(0, 1.05)

# By x-distribution
ax = axes[1]
x_dists = sorted(vo['x_distribution'].unique())
det_by_xd = []
for xd in x_dists:
    sub = vo[vo['x_distribution'] == xd]
    det_by_xd.append(float((sub['classification'] == 'detectable').mean()))
ax.bar(range(len(x_dists)), det_by_xd, color='#f59e0b', alpha=0.7)
ax.set_xticks(range(len(x_dists)))
ax.set_xticklabels([xd[:8] for xd in x_dists], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Detection Rate')
ax.set_title('Variance-Only Detection by X-Distribution')
ax.set_ylim(0, 1.05)

plt.suptitle('Variance-Only Detection Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig8_variance_only_detection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Overall variance-only detection rate: {(vo["classification"]=="detectable").mean():.1%}')

## 9. Metric Group Ablation

Compare joint-test p-values when different metric groups are removed.
Uses the individual metric p-values to simulate ablation without re-running permutation.

In [ ]:
# Group z-score columns by metric type
metric_groups = {
    'Correlation': [c for c in z_cols if any(k in c for k in ['pearson', 'spearman', 'covariance'])],
    'Distance': [c for c in z_cols if any(k in c for k in ['dcov', 'dcor'])],
    'Slopes': [c for c in z_cols if any(k in c for k in ['_ep_', '_pf_', 'seg_strength'])],
    'Bin-based': [c for c in z_cols if any(k in c for k in ['bin_eta', 'bin_amp', 'bin_bw'])],
    'Distribution': [c for c in z_cols if any(k in c for k in ['dist_ks', 'dist_wass'])],
    'MINE': [c for c in z_cols if any(k in c for k in ['mic', 'mas', 'mev', 'mcn'])],
    'LOWESS': [c for c in z_cols if 'lowess' in c],
}

# For each group, compute detection rate using max-Z of remaining metrics
signal_cases = df[df['category'].isin(['mean_only', 'mean+variance'])].copy()
z_matrix_full = signal_cases[z_cols].fillna(0).values

full_max_z = z_matrix_full.max(axis=1)
full_det_rate = float((signal_cases['classification'] == 'detectable').mean())

ablation_results = [{'group': 'All metrics', 'detection_rate': full_det_rate, 'n_metrics': len(z_cols)}]

for group_name, group_cols in metric_groups.items():
    remaining = [c for c in z_cols if c not in group_cols]
    if len(remaining) == 0:
        continue
    # This is an approximation: max-Z of remaining columns
    # (actual p-value would require re-running the joint test with the correct null)
    remaining_max_z = signal_cases[remaining].fillna(0).values.max(axis=1)
    # Use full T_joint threshold from original results as proxy
    ablation_results.append({
        'group': f'Without {group_name}',
        'detection_rate': np.nan,  # placeholder — exact needs re-permutation
        'n_metrics': len(remaining),
        'removed': len(group_cols),
        'mean_max_z_drop': float((full_max_z - remaining_max_z).mean()),
    })

abl_df = pd.DataFrame(ablation_results)
display(abl_df)

# Show which groups most often contribute the max Z
print('\n=== Max-Z Contribution by Metric Group ===')
z_matrix_df = signal_cases[z_cols].fillna(0)
max_col = z_matrix_df.idxmax(axis=1)
for group_name, group_cols in metric_groups.items():
    frac = max_col.isin(group_cols).mean()
    print(f'  {group_name:15s}: {frac:.1%} of cases have max-Z from this group')

## Summary

In [ ]:
print('=' * 60)
print('PERMUTATION TEST SUMMARY')
print('=' * 60)
print(f'Total cases: {len(df):,}')
print(f'Z-score metrics in joint test: {len(z_cols)}')
print()

for cat in CAT_ORDER:
    sub = df[df['category'] == cat]
    det = (sub['classification'] == 'detectable').mean()
    print(f'{CAT_LABELS[cat]:18s}: {len(sub):>7,} cases, detection rate = {det:.1%}')

print()
tn = df[df['category'] == 'true_null']
print(f'True Null FPR (α=0.05): {(tn["p_value"] <= 0.05).mean():.4f}')
print(f'True Null FPR (α=0.01): {(tn["p_value"] <= 0.01).mean():.4f}')